## load and prepare data

In [ ]:
%cd ../..
%matplotlib inline

import numpy as np
import pandas as pd
import statsmodels.stats.contingency_tables as sm_contigency_tables
from scipy.stats import mannwhitneyu

import sys
sys.path.insert(0, 'evaluation_meldgraph')
from vol_eval_plots import load_and_prepare_data, select_fcd_control_groups


In [ ]:
# 01_tables reports the subject counts before the FCD / control selection, so it applies that
# selection itself, further down
eval_stats_df = load_and_prepare_data(select_fcd_groups=False)

In [ ]:
# number of subjects per group and site for all subjects (before filtering for FCD, HC)
demographics_table = eval_stats_df.groupby(['group'])['site_subj_id'].nunique()
demographics_table = pd.DataFrame(demographics_table).reset_index().rename(columns={'group': 'description',
                                    'site_subj_id': 'number of subjects'})
demographics_table['category'] = 'group'
demographics_table['percentage'] = demographics_table['number of subjects'] / demographics_table['number of subjects'].sum() * 100
demographics_table[['category', 'description', 'number of subjects', 'percentage']]

In [ ]:
# number of subjects per site for all subjects (before filtering for FCD, HC)
demographics_table = eval_stats_df.groupby(['site'])['site_subj_id'].nunique()
demographics_table = pd.DataFrame(demographics_table).reset_index().rename(columns={'site': 'description',
                                    'site_subj_id': 'number of subjects'})
demographics_table['category'] = 'site'
demographics_table['percentage'] = demographics_table['number of subjects'] / demographics_table['number of subjects'].sum() * 100
demographics_table[['category', 'description', 'number of subjects', 'percentage']]

In [ ]:
eval_stats_df = select_fcd_control_groups(eval_stats_df)

In [ ]:
# show all unique site_subj_ids for subjects with category_FCD == 1 that have num_gt_clusters > 1
eval_stats_df[(eval_stats_df['category_FCD'] == 1) & (eval_stats_df['num_gt_clusters'] > 1)]['site_subj_id'].unique()

## demographic and clinical characteristics

In [ ]:
# now build a table with basic demographic data
stats_table = pd.DataFrame(columns=['category','description', 'number of subjects', 'percentage'])

In [ ]:
# number of subjects per group
demographics_table = eval_stats_df.groupby(['group'])['site_subj_id'].nunique()
demographics_table = pd.DataFrame(demographics_table).reset_index().rename(columns={'group': 'description',
                                    'site_subj_id': 'number of subjects'})
demographics_table['description'] = demographics_table['description'].replace({'control': 'Healthy control', 'patient': 'FCD'})
demographics_table['description'] = pd.Categorical(demographics_table['description'], categories=['FCD', 'Healthy control'], ordered=True)
demographics_table.sort_values('description', inplace=True)
demographics_table['category'] = 'Group'
demographics_table['percentage'] = demographics_table['number of subjects'] / demographics_table['number of subjects'].sum() * 100
demographics_table.rename
stats_table = pd.concat([stats_table, demographics_table], axis=0).reset_index(drop=True)
demographics_table[['category', 'description', 'number of subjects', 'percentage']]

In [ ]:
# number of subjects per site, where in group put 'control' and 'disease_control' together as 'control'
df_fcd_histopathology = eval_stats_df.replace({'group': {'control': 'control', 'disease_control': 'control'}})
demographics_table = df_fcd_histopathology.groupby(['site'])['site_subj_id'].nunique().reset_index()
demographics_table = pd.DataFrame(demographics_table).reset_index(drop=True).rename(columns={'site': 'description',
                                                                   'site_subj_id': 'number of subjects'})
demographics_table['category'] = 'Site'
demographics_table['percentage'] = demographics_table['number of subjects'] / demographics_table['number of subjects'].sum() * 100
stats_table = pd.concat([stats_table, demographics_table], axis=0).reset_index(drop=True)
demographics_table[['category', 'description', 'number of subjects', 'percentage']]

In [ ]:
# age group (< 18 vs >= 18 years)
df_fcd_histopathology = eval_stats_df.copy()
df_fcd_histopathology['age_group'] = df_fcd_histopathology['age_years'].apply(lambda x: 'Adult' if x >= 18 else 'Paediatric')
demographics_table = df_fcd_histopathology.groupby(['age_group'])['site_subj_id'].nunique().reset_index()
demographics_table = pd.DataFrame(demographics_table).reset_index(drop=True).rename(columns={'age_group': 'description',
                                                                   'site_subj_id': 'number of subjects'})
demographics_table['category'] = 'Age group'
demographics_table['percentage'] = demographics_table['number of subjects'] / demographics_table['number of subjects'].sum() * 100
stats_table = pd.concat([stats_table, demographics_table], axis=0).reset_index(drop=True)
demographics_table[['category', 'description', 'number of subjects', 'percentage']]

In [ ]:
# category_3T_MR_negative
df_fcd_histopathology = eval_stats_df.copy()
# only fcd patients
df_fcd_histopathology = df_fcd_histopathology[df_fcd_histopathology['category_FCD'].notna()]
df_fcd_histopathology.fillna({'category_3T_MR_negative': 'n/a', 'category_7T_MR_negative': 'n/a'}, inplace=True)

# add column 'description' that combines category_3T_MR_negative and category_7T_MR_negative
def assign_description_3T_7T_radiological(row):
    if row['category_3T_MR_negative'] == 0:
        return '3T MRI positive'
    else:
        if row['category_7T_MR_negative'] == 0:
            return '3T MRI negative, 7T MRI positive'
        else:
            return '3T and 7T MRI negative'

df_fcd_histopathology['description'] = df_fcd_histopathology.apply(assign_description_3T_7T_radiological, axis=1)
demographics_table = df_fcd_histopathology.groupby(['description'])['site_subj_id'].nunique().reset_index()
# order the descriptions in the order of '3T MRI positive', '3T MRI negative, 7T MRI positive', '3T and 7T MRI negative'
demographics_table['description'] = pd.Categorical(demographics_table['description'], categories=['3T MRI positive', '3T MRI negative, 7T MRI positive', '3T and 7T MRI negative'], ordered=True)
demographics_table.sort_values('description', inplace=True)
demographics_table = pd.DataFrame(demographics_table).reset_index(drop=True).rename(columns={'site_subj_id': 'number of subjects'})
demographics_table['category'] = 'Radiological MRI findings (FCD patients)'
demographics_table['percentage'] = demographics_table['number of subjects'] / demographics_table['number of subjects'].sum() * 100
stats_table = pd.concat([stats_table, demographics_table], axis=0).reset_index(drop=True)
demographics_table[['category', 'description', 'number of subjects', 'percentage']]

In [ ]:
eval_stats_df[eval_stats_df['category_FCD'] == 1].drop_duplicates(subset='site_subj_id')['category_7T_MR_negative'].value_counts()

In [ ]:
# surgery performed
df_fcd_histopathology = eval_stats_df.copy()
# only fcd patients
df_fcd_histopathology = df_fcd_histopathology[df_fcd_histopathology['category_FCD'].notna()]
df_fcd_histopathology['surgery'] = df_fcd_histopathology.apply(lambda row: 'Resection' if row['surgery_resection'] == 1 else ('Ablation' if row['surgery_ablation'] == 1 else 'None'), axis=1)
demographics_table = df_fcd_histopathology.groupby('surgery')['site_subj_id'].nunique().reset_index()
demographics_table = pd.DataFrame(demographics_table).reset_index(drop=True).rename(columns={'surgery': 'description',
                                                                   'site_subj_id': 'number of subjects'})
demographics_table['category'] = 'Surgery'
demographics_table['percentage'] = demographics_table['number of subjects'] / demographics_table['number of subjects'].sum() * 100
# sort by order none, resection, ablation
demographics_table['description'] = pd.Categorical(demographics_table['description'], categories=['Resection', 'Ablation', 'None'], ordered=True)
demographics_table = demographics_table.sort_values('description').reset_index(drop=True)
stats_table = pd.concat([stats_table, demographics_table], axis=0).reset_index(drop=True)
demographics_table[['category', 'description', 'number of subjects', 'percentage']]

In [ ]:
# confirmed by histopathology (only for operated FCD patients)
df_fcd_histopathology = eval_stats_df.copy()
# only fcd patients that underwent surgery
df_fcd_histopathology.fillna({'surgery_resection': 0, 'surgery_ablation': 0, 'confirmed_histology': 'n/a'}, inplace=True)
df_fcd_histopathology = df_fcd_histopathology[(df_fcd_histopathology['category_FCD'] == 1) & ((df_fcd_histopathology['surgery_resection'] == 1))]
demographics_table = df_fcd_histopathology.groupby('confirmed_histology')['site_subj_id'].nunique().reset_index()
demographics_table = pd.DataFrame(demographics_table).reset_index(drop=True).rename(columns={'confirmed_histology': 'description',
                                                                   'site_subj_id': 'number of subjects'})
demographics_table['category'] = 'Confirmed by histopathology'
demographics_table['percentage'] = demographics_table['number of subjects'] / demographics_table['number of subjects'].sum() * 100
# convert description to yes/no
demographics_table['description'] = demographics_table['description'].replace({1: 'Yes', 0: 'No', 'n/a': 'Not available'})
stats_table = pd.concat([stats_table, demographics_table], axis=0).reset_index(drop=True)
demographics_table[['category', 'description', 'number of subjects', 'percentage']]

In [ ]:
df_fcd_histopathology['site_subj_id'].unique()

In [ ]:
# favorable (Engel I at 1 year) outcome (only for operated or ablated HS patients)
df_fcd_histopathology = eval_stats_df.copy()
# only hs patients that underwent surgery
df_fcd_histopathology.fillna({'surgery_resection': 0, 'surgery_ablation': 0, 'favorable_outcome': 'n/a'}, inplace=True)
df_fcd_histopathology = df_fcd_histopathology[(df_fcd_histopathology['category_FCD'] == 1) & ((df_fcd_histopathology['surgery_resection'] == 1) | (df_fcd_histopathology['surgery_ablation'] == 1))]
demographics_table = df_fcd_histopathology.groupby('favorable_outcome')['site_subj_id'].nunique().reset_index()
demographics_table = pd.DataFrame(demographics_table).reset_index(drop=True).rename(columns={'favorable_outcome': 'description',
                                                                   'site_subj_id': 'number of subjects'})
demographics_table['category'] = 'Favourable outcome'
demographics_table['percentage'] = demographics_table['number of subjects'] / demographics_table['number of subjects'].sum() * 100
# convert description to yes/no
demographics_table['description'] = demographics_table['description'].replace({1: 'Yes',
                                                     0.5: 'Less than 1 year follow-up',
                                                     0: 'No',
                                                     'n/a': 'Not available'})
demographics_table['description'] = pd.Categorical(demographics_table['description'], categories=['Yes', 'Less than 1 year follow-up', 'No', 'Not available'], ordered=True)
demographics_table.sort_values('description', inplace=True)
stats_table = pd.concat([stats_table, demographics_table], axis=0).reset_index(drop=True)
demographics_table[['category', 'description', 'number of subjects', 'percentage']]

### Table 1

In [ ]:
# display stats table and round percentage to 1 decimal
stats_table['percentage'] = stats_table['percentage'].round(1)
stats_table[['category', 'description', 'number of subjects', 'percentage']]

## additional data (not in the demographics summary table)

In [ ]:
# age 
df_demographics = eval_stats_df.copy()
df_demographics = df_demographics.drop_duplicates(subset=['site_subj_id'])
table = df_demographics['age_years'].describe()
table['median'] = df_demographics['age_years'].median()
table


In [ ]:
df_demographics = eval_stats_df.copy()
df_demographics = df_demographics.drop_duplicates(subset=['site_subj_id'])
df_demographics['sex'].value_counts()

In [ ]:
# test whether there is a relationship between sex and the groups
contingency_table = pd.crosstab(df_demographics['sex'], df_demographics['group'])
table = sm_contigency_tables.Table(contingency_table)
print(contingency_table)
print(f"chi-square test for sex vs group p-value: {table.test_nominal_association().pvalue}")

In [ ]:
# test whether the age distribution is different between sites groups (kruskal-wallis test)
df_demographics = eval_stats_df.copy()
df_demographics = df_demographics.drop_duplicates(subset=['site_subj_id'])
groups = df_demographics.groupby('group')['age_years'].apply(list)
groups

In [ ]:
# compare age distribution between groups using Mann-Whitney U test for non-parametric comparison
from scipy.stats import shapiro
for group_name, group_data in groups.items():
    stat, p_value = shapiro(group_data)
    print(f"Shapiro-Wilk test for {group_name}: statistic={stat:.4f}, p-value={p_value:.4f}")
from scipy.stats import mannwhitneyu

res = mannwhitneyu(groups['control'], groups['patient'], alternative='two-sided')

print(f"Mann-Whitney-U test for age vs group p-value: {res.pvalue}")

In [ ]:
eval_stats_df.drop_duplicates(subset=['site_subj_id'])['7T_T1w_type'].value_counts()

In [ ]:
eval_stats_df.drop_duplicates(subset=['site_subj_id'])['7T_T1w_pTx'].value_counts()

In [ ]:
# min voxel size
df_input_stats = eval_stats_df.copy()
df_input_stats.dropna(subset=['input_path'], inplace=True)
# T1w
print(df_input_stats[df_input_stats['analysis_group'].isin(['3T', '7T default', '7T adapted'])].groupby(['analysis_group'])['T1w_min_voxel_size'].value_counts())
print(df_input_stats[df_input_stats['analysis_group'].isin(['3T', '7T default', '7T adapted'])].groupby(['analysis_group'])['T1w_min_voxel_size'].median())

# select matched FLAIR subjects
subjs_3T_FLAIR = df_input_stats[df_input_stats['analysis_group'] == '3T FLAIR']['site_subj_id'].unique()
subjs_7T_FLAIR = df_input_stats[df_input_stats['analysis_group'] == '7T adapted FLAIR']['site_subj_id'].unique()
df_input_stats_FLAIR = df_input_stats[df_input_stats['site_subj_id'].isin(np.intersect1d(subjs_3T_FLAIR, subjs_7T_FLAIR))].copy()
print(df_input_stats_FLAIR[df_input_stats_FLAIR['analysis_group'].isin(['3T FLAIR', '7T adapted FLAIR'])].groupby(['analysis_group'])['FLAIR_min_voxel_size'].value_counts())
print(df_input_stats_FLAIR[df_input_stats_FLAIR['analysis_group'].isin(['3T FLAIR', '7T adapted FLAIR'])].groupby(['analysis_group'])['FLAIR_min_voxel_size'].median())